In [2]:
%load_ext autoreload
%autoreload 2

from src.dataset import *
from src.preprocess import *
from src.pipeline import *

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# sns.set_style("darkgrid")

%matplotlib inline

In [53]:
np.random.seed(24)

data_generator = DatasetGenerator()
feature_specs = {
    "age": {
        "type": FeatureType.NUMERIC,
        "params": {
            "distribution": "normal",
            "mean": 35,
            "std": 10
        }
    },
    "years_experience": {
        "type": FeatureType.NUMERIC,
        "params": {
            "distribution": "normal",
            "mean": 8,
            "std": 5
        }
    },
    "education_level": {
        "type": FeatureType.CATEGORICAL,
        "params": {
            "categories": ["high_school", "bachelors", "masters", "phd"],
            "probabilities": [0.3, 0.4, 0.2, 0.1]
        }
    }
}


df = data_generator.generate_synthetic_dataset(
    n_samples=10000,
    feature_specs=feature_specs,
    protected_attr_ratio=0.5,  # 50-50 split between protected groups
    positive_label_ratio=0.6   # 60% positive labels (hired)
)

In [ ]:
# df.to_csv("data/test_data.csv", index=False)

In [4]:
df["protected_attribute"].value_counts() / df.shape[0]

protected_attribute
0    0.5057
1    0.4943
Name: count, dtype: float64

In [5]:
df["education_level"].value_counts() / df.shape[0]

education_level
bachelors      0.3979
high_school    0.2946
masters        0.2085
phd            0.0990
Name: count, dtype: float64

In [6]:
df.columns

Index(['protected_attribute', 'age', 'years_experience', 'education_level',
       'target'],
      dtype='object')

In [7]:
repr_bias = RepresentationBias(protected_attribute="protected_attribute", underrepresented_group=1, reduction_factor=0.5)
temp_df = repr_bias.transform(df)

temp_df["protected_attribute"].value_counts() / temp_df.shape[0]

protected_attribute
0    0.67167
1    0.32833
Name: count, dtype: float64

In [8]:
temp_df.head()

,protected_attribute,age,years_experience,education_level,target
0,1,28.768325,1.244733,masters,0
1,1,14.182482,3.702951,high_school,1
3,0,31.550724,18.297600,high_school,1
4,0,27.154984,9.053062,masters,0
5,1,33.248210,9.263200,masters,0


In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [48]:
pipeline = Pipeline([
    # ('representation_bias', RepresentationBias(protected_attribute="protected_attribute", underrepresented_group=1, reduction_factor=0.5)),
    ('education_encoder', FeatureEncoder(method="label", column="education_level")),
    ('model', RandomForestClassifier())
])

In [49]:
pipeline.fit(df, target_column='target')

In [50]:
preds = pipeline.predict(df)

In [51]:
sum(preds)

np.int32(6038)

In [52]:
pipeline.score(df, target_column="target")

1.0